
# QML-SleepNet — FINAL Fixed Equal-Logit Ensemble Post-hoc x Score v1

This notebook does **no fitting and no selection**.

It only:

1. verifies the SHA-256 of the already-frozen fixed ensemble predictions;
2. opens both official x annotation sources;
3. verifies exact annotation agreement;
4. scores the fixed prediction artifact;
5. compares it with the previously frozen Strict Bridge result.

Scientific status remains:

**post-hoc extended-pipeline official x-set evidence**

because official x outcomes were historically visible before this exact fixed ensemble was proposed.


In [1]:

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    confusion_matrix
)

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
RAW = ROOT / "data/raw/apnea_ecg"

OUT = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "FINAL_FIXED_EQUAL_LOGIT_ENSEMBLE_v1"
)

FROZEN = (
    OUT / "FINAL_FIXED_EQUAL_LOGIT_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
)
FREEZE = OUT / "FINAL_FIXED_EQUAL_LOGIT_PRE_SCORE_FREEZE_MANIFEST.json"

LABEL_JSON = RAW / "test_set_apnea_labels.json"
LABEL_TXT = RAW / "test-dataset-annos.txt"

BRIDGE_METRICS = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX/STRICT_BRIDGE_FINAL_v1"
    / "STRICT_BRIDGE_POSTHOC_X_METRICS.json"
)

for p in [FROZEN, FREEZE, LABEL_JSON, LABEL_TXT, BRIDGE_METRICS]:
    if not p.is_file():
        raise FileNotFoundError(p)

print("Scoring frozen fixed ensemble only.")
print("No fitting. No reselection. No threshold change.")


Mounted at /content/drive
Scoring frozen fixed ensemble only.
No fitting. No reselection. No threshold change.


In [2]:

def sha256_file(path, chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_json(path,obj):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,default=str))
    tmp.replace(path)

def normalize_json_labels(obj):
    out={}
    for rec,val in obj.items():
        seq=val.strip() if isinstance(val,str) else "".join(map(str,val))
        if set(seq)-{"A","N"}:
            raise RuntimeError(f"{rec}: unexpected symbols")
        out[str(rec)]=seq
    return out

def parse_txt(path):
    out={}
    current=None
    chunks=[]

    for raw in Path(path).read_text().splitlines():
        line=raw.strip()
        if not line:
            continue

        if len(line)==3 and line.startswith("x") and line[1:].isdigit():
            if current is not None:
                out[current]="".join(chunks)
            current=line
            chunks=[]
            continue

        if current is None:
            continue

        parts=line.split()
        if len(parts)>=2 and parts[0].isdigit():
            seq="".join(parts[1:]).strip()
            if set(seq)-{"A","N"}:
                raise RuntimeError(line)
            chunks.append(seq)

    if current is not None:
        out[current]="".join(chunks)

    return out

def metrics(y,score,threshold=0.5):
    y=np.asarray(y,dtype=np.int8)
    score=np.asarray(score,dtype=np.float64)

    pred=(score>=threshold).astype(np.int8)

    tn,fp,fn,tp=confusion_matrix(
        y,pred,labels=[0,1]
    ).ravel()

    return {
        "n":int(len(y)),
        "positives":int(y.sum()),
        "negatives":int((1-y).sum()),
        "accuracy":float(accuracy_score(y,pred)),
        "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
        "precision":float(precision_score(y,pred,zero_division=0)),
        "sensitivity":float(recall_score(y,pred,zero_division=0)),
        "specificity":float(tn/max(tn+fp,1)),
        "f1":float(f1_score(y,pred,zero_division=0)),
        "mcc":float(matthews_corrcoef(y,pred)),
        "auroc":float(roc_auc_score(y,score)),
        "auprc":float(average_precision_score(y,score)),
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
    }


In [3]:

# VERIFY FREEZE BEFORE LABEL OPEN

freeze=json.loads(FREEZE.read_text())

if freeze.get("official_x_labels_read_in_this_notebook") is not False:
    raise RuntimeError("Freeze provenance failure")
if freeze.get("official_x_labels_used_for_selection") is not False:
    raise RuntimeError("Freeze provenance failure")

actual_sha=sha256_file(FROZEN)

if actual_sha != freeze["prediction_sha256"]:
    raise RuntimeError("Frozen prediction SHA mismatch")

z=np.load(FROZEN,allow_pickle=False)

UID=np.asarray(z["test_uids"]).astype(str)
SCORE=np.asarray(z["ensemble_hmm_logit_mean"],dtype=np.float64)
PRED=np.asarray(z["prediction"],dtype=np.int8)

THR=float(np.asarray(z["hard_threshold"]).item())

if len(UID)!=17248:
    raise RuntimeError(f"Expected 17,248 rows, got {len(UID)}")

print("Freeze SHA verified:",actual_sha)
print("Rows:",len(UID))
print("Threshold:",THR)
print("Branches:",np.asarray(z["branch_order"]).astype(str).tolist())
print("Weights:",np.asarray(z["weights"],float).tolist())
print("NOW official labels may be opened solely for scoring.")


Freeze SHA verified: f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef
Rows: 17248
Threshold: 0.5
Branches: ['bridge', 'qt_angle_rx', 'current_stage06']
Weights: [0.3333333333333333, 0.3333333333333333, 0.3333333333333333]
NOW official labels may be opened solely for scoring.


In [4]:

# OPEN + CROSS-CHECK OFFICIAL LABEL SOURCES

j=normalize_json_labels(json.loads(LABEL_JSON.read_text()))
t=parse_txt(LABEL_TXT)

records=[f"x{i:02d}" for i in range(1,36)]

if sorted(j)!=records or sorted(t)!=records:
    raise RuntimeError("Official record universe mismatch")

for rec in records:
    if j[rec]!=t[rec]:
        raise RuntimeError(f"Official label source disagreement: {rec}")

label_by_uid={
    f"{rec}:{i}":1 if ch=="A" else 0
    for rec in records
    for i,ch in enumerate(j[rec])
}

missing=[u for u in UID if u not in label_by_uid]

if missing:
    raise RuntimeError(f"Frozen UID missing annotation: {missing[:10]}")

Y=np.asarray([label_by_uid[u] for u in UID],dtype=np.int8)

marker={
    "status":"FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_SCORING",
    "scored_at_utc":datetime.now(timezone.utc).isoformat(),
    "prediction_sha256":sha256_file(FROZEN),
    "method_changed_after_label_open":False,
    "scientific_status":"post-hoc extended-pipeline official x-set evidence",
}

atomic_json(
    OUT / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_SCORING_MARKER.json",
    marker
)

print("Official JSON/TXT exact agreement: PASS")
print("Scoring rows:",len(Y))


Official JSON/TXT exact agreement: PASS
Scoring rows: 17248


In [5]:

FINAL=metrics(Y,SCORE,THR)

FINAL.update({
    "pipeline":(
        "Bridge + QT Angle-Rx + corrected Stage06; "
        "each -> NLL temperature -> prior-corrected HMM lambda=1; "
        "fixed equal mean of HMM posterior logits -> threshold 0.5"
    ),
    "weights":[1/3,1/3,1/3],
    "threshold":THR,
    "scientific_status":"post-hoc extended-pipeline official x-set evidence",
})

atomic_json(
    OUT / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_METRICS.json",
    FINAL
)

pd.DataFrame({
    "uid":UID,
    "y_true":Y,
    "ensemble_score":SCORE,
    "prediction":PRED,
}).to_csv(
    OUT / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_PREDICTIONS.csv",
    index=False
)

print("="*110)
print("FINAL FIXED EQUAL-LOGIT ENSEMBLE — POST-HOC OFFICIAL X RESULT")
print("="*110)
print(json.dumps(FINAL,indent=2))


FINAL FIXED EQUAL-LOGIT ENSEMBLE — POST-HOC OFFICIAL X RESULT
{
  "n": 17248,
  "positives": 6547,
  "negatives": 10701,
  "accuracy": 0.9011479591836735,
  "balanced_accuracy": 0.9001158045109892,
  "precision": 0.8514808362369338,
  "sensitivity": 0.8958301512142967,
  "specificity": 0.9044014578076816,
  "f1": 0.8730926684034239,
  "mcc": 0.7929076491751258,
  "auroc": 0.9620666432037354,
  "auprc": 0.9427717592257047,
  "tn": 9678,
  "fp": 1023,
  "fn": 682,
  "tp": 5865,
  "pipeline": "Bridge + QT Angle-Rx + corrected Stage06; each -> NLL temperature -> prior-corrected HMM lambda=1; fixed equal mean of HMM posterior logits -> threshold 0.5",
  "weights": [
    0.3333333333333333,
    0.3333333333333333,
    0.3333333333333333
  ],
  "threshold": 0.5,
  "scientific_status": "post-hoc extended-pipeline official x-set evidence"
}


In [6]:

# Compare to the previously locked Strict Bridge result.

bridge=json.loads(BRIDGE_METRICS.read_text())

keys=[
    "accuracy","balanced_accuracy","precision","sensitivity",
    "specificity","f1","mcc","auroc","auprc"
]

delta={
    k:float(FINAL[k]-float(bridge[k]))
    for k in keys
}

comparison={
    "fixed_equal_logit_ensemble":FINAL,
    "strict_bridge":bridge,
    "delta_ensemble_minus_bridge":delta,
}

atomic_json(
    OUT / "FINAL_FIXED_EQUAL_LOGIT_VS_STRICT_BRIDGE.json",
    comparison
)

print("DELTA FIXED ENSEMBLE - STRICT BRIDGE")
for k,v in delta.items():
    print(f"{k:20s}: {v:+.6f} ({v*100:+.3f} pp)")


DELTA FIXED ENSEMBLE - STRICT BRIDGE
accuracy            : +0.007189 (+0.719 pp)
balanced_accuracy   : +0.002355 (+0.235 pp)
precision           : +0.025832 (+2.583 pp)
sensitivity         : -0.017718 (-1.772 pp)
specificity         : +0.022428 (+2.243 pp)
f1                  : +0.005715 (+0.572 pp)
mcc                 : +0.010723 (+1.072 pp)
auroc               : +0.004172 (+0.417 pp)
auprc               : +0.003096 (+0.310 pp)


In [7]:

# Per-record diagnostics + record-macro summary.

rec=np.asarray([u.rsplit(":",1)[0] for u in UID])

rows=[]

for r in sorted(np.unique(rec)):
    idx=np.where(rec==r)[0]
    yy=Y[idx]
    ss=SCORE[idx]
    pp=(ss>=THR).astype(np.int8)

    tn,fp,fn,tp=confusion_matrix(
        yy,pp,labels=[0,1]
    ).ravel()

    row={
        "record_name":r,
        "n":int(len(idx)),
        "positives":int(yy.sum()),
        "accuracy":float(accuracy_score(yy,pp)),
        "f1":float(f1_score(yy,pp,zero_division=0)),
        "sensitivity":float(recall_score(yy,pp,zero_division=0)),
        "specificity":float(tn/max(tn+fp,1)),
    }

    if len(np.unique(yy))==2:
        row["balanced_accuracy"]=float(balanced_accuracy_score(yy,pp))
        row["auroc"]=float(roc_auc_score(yy,ss))
        row["auprc"]=float(average_precision_score(yy,ss))
    else:
        row["balanced_accuracy"]=np.nan
        row["auroc"]=np.nan
        row["auprc"]=np.nan

    rows.append(row)

rdf=pd.DataFrame(rows)

rdf.to_csv(
    OUT / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_PER_RECORD_METRICS.csv",
    index=False
)

macro={
    "record_macro_accuracy":float(rdf["accuracy"].mean()),
    "record_macro_f1":float(rdf["f1"].mean()),
    "record_macro_balanced_accuracy_valid":float(rdf["balanced_accuracy"].dropna().mean()),
    "record_macro_auroc_valid":float(rdf["auroc"].dropna().mean()),
    "record_macro_auprc_valid":float(rdf["auprc"].dropna().mean()),
}

atomic_json(
    OUT / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_RECORD_MACRO.json",
    macro
)

print("\nRECORD-MACRO")
print(json.dumps(macro,indent=2))



RECORD-MACRO
{
  "record_macro_accuracy": 0.9026254240778879,
  "record_macro_f1": 0.567557944438842,
  "record_macro_balanced_accuracy_valid": 0.7556882011858456,
  "record_macro_auroc_valid": 0.8918519455361159,
  "record_macro_auprc_valid": 0.7112754708398232
}



## Freeze rule after scoring

Whatever this scorer prints is the fixed ensemble's result.

Do **not** alter weights, branch membership, temperatures, HMM parameters, λ, or threshold after seeing it.

If the expected gain reproduces, this closes Task A metric development permanently.
